# QMUL-SurvFace-v1 공식 매니페스트 생성

공식 open-set face identification 평가 파일을 기준으로 gallery, mated probe, unmated probe를 분리합니다. `gallery_img_ID_pairs.mat`와 `mated_probe_img_ID_pairs.mat`의 행 순서와 ID를 그대로 보존합니다.

중요: SurvFace 공식 평가 코드는 3,000개 등록 ID의 gallery feature를 identity별 평균 template로 만들며, unmated probe를 모두 미등록으로 평가합니다. 현재 저장소의 일반 `00_protocol_and_run_freeze.ipynb`는 gallery를 다시 표본추출하고 known unknown을 요구하므로, 이 매니페스트를 그대로 넣어 **SurvFace 공식 결과라고 주장하면 안 됩니다**. 아래 역할별 CSV를 읽는 공식 프로토콜 어댑터가 필요합니다.

`WRITE_OUTPUTS=False`여도 MAT 구조, MAT와 실제 파일의 일치, ID 집합, 중복, 전체 역할을 모두 검사합니다. 파일만 저장하지 않습니다.

## 중단 후 재시작

- MAT/이미지 검사 셀 이전 또는 도중에 중단: 커널을 재시작하고 첫 코드 셀부터 다시 실행합니다.
- 저장 셀에서 중단: 출력 폴더를 확인한 뒤 `OVERWRITE=True`로 바꾸고 첫 코드 셀부터 다시 실행합니다.
- `training_set`은 공식 test protocol과 identity 관계가 확정되지 않았으므로 이 매니페스트에 섞지 않습니다.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import scipy
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. D:/ronbun 안에서 노트북을 실행하십시오.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import (
    build_survface_official_manifest,
    write_survface_official_bundle,
)

print(f'project_root: {PROJECT_ROOT}')
print(f'python: {sys.executable}')
print(f'pandas: {pd.__version__}, scipy: {scipy.__version__}')


## 1. 경로와 저장 모드 고정

데이터셋 루트에는 `Face_Identification_Evaluation`과 `Face_Identification_Test_Set`이 모두 있어야 합니다.


In [ ]:
WRITE_OUTPUTS = False  # 검증 완료 후에만 True
OVERWRITE = False      # 기존 결과를 의도적으로 교체할 때만 True

SURVFACE_ROOT = PROJECT_ROOT / 'data/raw/QMUL-SurvFace'
OUTPUT_DIR = PROJECT_ROOT / 'data/interim/survface'

display(pd.Series({
    'WRITE_OUTPUTS': WRITE_OUTPUTS,
    'OVERWRITE': OVERWRITE,
    'SURVFACE_ROOT': str(SURVFACE_ROOT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
}, name='value').to_frame())


## 2. 공식 MAT와 전체 이미지 검사

gallery와 mated probe는 MAT의 filename/ID/order를 사용합니다. MAT에 빠졌거나 폴더에만 추가로 존재하는 파일이 있으면 중단합니다. unmated probe는 공식 true identity가 없으므로 파일별로 불투명한 synthetic ID를 부여하며, identity 추정에 파일명을 사용하지 않습니다.


In [ ]:
bundle = build_survface_official_manifest(SURVFACE_ROOT, PROJECT_ROOT)

display(pd.Series(bundle.summary, name='value').to_frame())
display(
    bundle.manifest.groupby(['protocol_role', 'probe_type']).agg(
        images=('image_id', 'size'),
        identities=('identity_id', 'nunique'),
    )
)
display(bundle.gallery.head(5))
display(bundle.registered_probes.head(5))
display(bundle.unknown_unknown_probes.head(5))
print('SurvFace 공식 매니페스트 검증 통과')


## 3. 역할별 결과 저장

`official_manifest.csv`는 전체 행을 포함하고, 나머지 CSV는 공식 역할별 입력입니다. gallery는 이미지 단위 행이며 실제 평가 시 같은 `official_identity_id`의 임베딩을 평균하여 template 하나로 만들어야 합니다.


In [ ]:
output_names = (
    'official_manifest.csv',
    'gallery.csv',
    'registered_probes.csv',
    'unknown_unknown_probes.csv',
    'gallery_identities.txt',
    'unknown_unknown_identities.txt',
    'summary.json',
)
planned_paths = {name: OUTPUT_DIR / name for name in output_names}

if WRITE_OUTPUTS:
    written_paths = write_survface_official_bundle(
        bundle, OUTPUT_DIR, overwrite=OVERWRITE
    )
    print('저장 완료')
else:
    written_paths = planned_paths
    print('WRITE_OUTPUTS=False: 검증만 완료했으며 파일은 저장하지 않았습니다.')

display(pd.DataFrame(
    [{'file': name, 'path': str(path), 'exists': path.is_file()}
     for name, path in written_paths.items()]
))


## 다음 단계

1. `WRITE_OUTPUTS=True`로 전체 재실행하여 `data/interim/survface/`의 역할별 CSV를 생성합니다.
2. ArcFace 추출 단계는 `gallery.csv`, `registered_probes.csv`, `unknown_unknown_probes.csv`의 `protocol_index` 순서를 보존합니다.
3. gallery 임베딩은 `official_identity_id`별 평균 template로 집계합니다.
4. 공식 평가는 mated probe와 unmated probe만 사용하고 known unknown 결과를 임의로 만들지 않습니다.
5. 일반 00 노트북을 연결하기 전, gallery 재표본추출 없이 `protocol_role`을 읽는 SurvFace 전용 어댑터를 구현해야 합니다.
